安装依赖

In [18]:
!pip install requests pandas tqdm -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


导入库与全局配置

In [ ]:
import requests
import os
import time
from tqdm import tqdm

# ====================== 必须修改 ======================
# SEC官方要求提供身份标识，替换为你的真实姓名和邮箱
USER_AGENT = "SEC官方要求提供身份标识，替换为你的真实姓名和邮箱"
# ======================================================

# SEC数据API请求头
HEADERS_DATA = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

# EDGAR文件下载请求头
HEADERS_EDGAR = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip, deflate"
}

# 文件保存根目录
SAVE_ROOT = "./10K_filings"
os.makedirs(SAVE_ROOT, exist_ok=True)

# 目标股票代码列表
# 以下为示例代码，可以换成你想要的任何公司，下载模块cpu运行速度大概1-3分钟，看你网速，只适合小规模下载哈
TARGET_TICKERS = ["ACCS", "ACU", "AEF", "AEON", "AIM", "AIRI", "AMBI", "AMBO", "AMS", "AMZE"]

获取 Ticker 与 CIK 的官方映射

In [ ]:
def get_ticker_cik_mapping():
    url = "https://www.sec.gov/files/company_tickers.json"
    # 单独构造对应域名的请求头，不能复用 data.sec.gov 的 Host
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Encoding": "gzip, deflate",
        "Host": "www.sec.gov"
    }
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    mapping = {}
    for item in data.values():
        ticker = item["ticker"].upper()
        cik = str(item["cik_str"]).zfill(10)
        mapping[ticker] = cik
    return mapping

ticker_to_cik = get_ticker_cik_mapping()
time.sleep(0.2)

获取指定公司 2000 年后的 10-K 提交记录

In [22]:
def fetch_10k_filings(cik, start_year=2000):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    resp = requests.get(url, headers=HEADERS_DATA)
    resp.raise_for_status()
    filings = resp.json()["filings"]["recent"]
    
    ten_k_list = []
    # 筛选10-K类型且年份符合要求的提交
    for form, date, acc, doc in zip(
        filings["form"],
        filings["filingDate"],
        filings["accessionNumber"],
        filings["primaryDocument"]
    ):
        filing_year = int(date.split("-")[0])
        if form == "10-K" and filing_year >= start_year:
            ten_k_list.append({
                "year": filing_year,
                "filing_date": date,
                "accession": acc,
                "document": doc
            })
    return ten_k_list

先测试单家公司下载单份 10-K 文件

In [24]:
def download_10k_file(cik, filing, save_dir):
    # 拼接EDGAR归档文件直链
    acc_clean = filing["accession"].replace("-", "")
    file_url = f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}/{acc_clean}/{filing['document']}"
    save_path = os.path.join(save_dir, f"{filing['year']}_{filing['document']}")
    
    # 已下载则跳过
    if os.path.exists(save_path):
        return True
    
    try:
        resp = requests.get(file_url, headers=HEADERS_EDGAR, timeout=30)
        resp.raise_for_status()
        with open(save_path, "wb") as f:
            f.write(resp.content)
        time.sleep(0.2)  # 控制请求频率
        return True
    except Exception as e:
        print(f"下载失败 {file_url}: {str(e)}")
        return False

批量执行 10 家公司的爬取

In [25]:
for ticker in tqdm(TARGET_TICKERS, desc="整体进度"):
    ticker_upper = ticker.upper()
    if ticker_upper not in ticker_to_cik:
        print(f"\n未找到 {ticker} 的CIK信息，跳过")
        continue
    
    cik = ticker_to_cik[ticker_upper]
    company_dir = os.path.join(SAVE_ROOT, ticker_upper)
    os.makedirs(company_dir, exist_ok=True)
    
    try:
        ten_k_list = fetch_10k_filings(cik)
    except Exception as e:
        print(f"\n获取 {ticker} 提交记录失败: {str(e)}")
        continue
    
    if not ten_k_list:
        print(f"\n{ticker} 无2000年后的10-K文件")
        continue
    
    print(f"\n{ticker} 找到 {len(ten_k_list)} 份10-K，开始下载...")
    success_count = 0
    for filing in ten_k_list:
        if download_10k_file(cik, filing, company_dir):
            success_count += 1
    
    print(f"{ticker} 下载完成：成功 {success_count}/{len(ten_k_list)} 份")
    time.sleep(0.5)

print("\n=== 全部任务执行完毕 ===")

整体进度:   0%|          | 0/10 [00:00<?, ?it/s]


ACCS 找到 19 份10-K，开始下载...
ACCS 下载完成：成功 19/19 份


整体进度:  10%|█         | 1/10 [00:53<08:04, 53.86s/it]


ACU 找到 24 份10-K，开始下载...
ACU 下载完成：成功 24/24 份


整体进度:  30%|███       | 3/10 [02:07<04:09, 35.68s/it]

...


AMS 找到 25 份10-K，开始下载...
AMS 下载完成：成功 25/25 份


整体进度:  90%|█████████ | 9/10 [03:56<00:21, 21.31s/it]


AMZE 找到 5 份10-K，开始下载...
AMZE 下载完成：成功 5/5 份


整体进度: 100%|██████████| 10/10 [04:02<00:00, 24.28s/it]


=== 全部任务执行完毕 ===
